# Transformers and Self-Attention for Particle Physics

**ML4FP Tutorial -- Notebook 1**

---


By the end you will:
1. Understand self-attention, the core mechanism behind transformers
2. Build a working transformer jet classifier

**References:**
- Vaswani et al., "Attention Is All You Need" ([arXiv:1706.03762](https://arxiv.org/abs/1706.03762))
- Qu et al., "Particle Transformer for Jet Tagging" ([arXiv:2202.03772](https://arxiv.org/abs/2202.03772))

---
## Section 1: 

### Particle Jets

When protons collide at the LHC, the hard scatter produces quarks and gluons that hadronize into collimated sprays of particles called **jets**. Each particle carries kinematic information ($p_T$, $\eta$, $\phi$, energy), and the goal of jet tagging is to determine what initiated the jet -- a Higgs boson decay, a top quark, or ordinary QCD radiation.

### Why Transformers?

Differences between language data and jet data:

| | Language | Particle Jets |
|:---|:---|:---|
| **Data** | "The cat sat on the mat" | {particle1, particle2, particle3, ...} |
| **Order** | Matters ("dog bites man" $\neq$ "man bites dog") | Does not matter (same jet regardless of particle ordering) |
| **Technical term** | Sequence | **Set** (permutation invariant) |

Transformers without positional encoding are **permutation equivariant** -- they treat their input as a set, not a sequence. NLP transformers add positional encoding to break this symmetry because word order can change the meaning of the sentence. For jets, we skip that step entirely, because natural set format of the data and matches that of the architecture

In [ ]:
%matplotlib inline

import os
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
## Section 2: Loading and Exploring the Data

In [ ]:
# Download the dataset if needed
from tutorial_data import ensure_jetclass_example

data_file = ensure_jetclass_example('../data')

In [ ]:
# Load the data
from dataloader import read_file

x_particles, x_jets, y = read_file(data_file)

print("=== Dataset shapes ===")
print(f"x_particles : {x_particles.shape}  (jets, features, max_particles)")
print(f"y (one-hot) : {y.shape}  (jets, classes)")
print(f"\nEach particle has 4 features: pT, eta, phi, energy")
print(f"Jets are zero-padded to {x_particles.shape[2]} particles")

In [ ]:
# The 10 jet classes
class_names = ['QCD', 'Hbb', 'Hcc', 'Hgg', 'H4q', 'Hqql', 'Zqq', 'Wqq', 'Tbqq', 'Tbl']
class_descriptions = [
    'Light quark/gluon (background)',
    'Higgs -> bb',
    'Higgs -> cc',
    'Higgs -> gg',
    'Higgs -> 4 quarks (via WW*/ZZ*)',
    'Higgs -> qqln',
    'Z -> qq',
    'W -> qq',
    'Top -> bqq',
    'Top -> bln',
]

print("Jet classes:")
for i, (name, desc) in enumerate(zip(class_names, class_descriptions)):
    count = y[:, i].sum()
    print(f"  {i}: {name:5s}  {desc:40s}  ({int(count):,} jets)")

In [ ]:
# Visualize 4 jets in the eta-phi plane
# Each dot = one particle, size proportional to pT

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

show_classes = [0, 1, 8, 4]  # QCD, Hbb, Tbqq, H4q
for ax_idx, cls in enumerate(show_classes):
    jet_i = np.where(y[:, cls] == 1)[0][0]
    pt  = x_particles[jet_i, 0, :]
    eta = x_particles[jet_i, 1, :]
    phi = x_particles[jet_i, 2, :]
    mask = pt > 0  # non-padded particles
    
    sc = axes[ax_idx].scatter(
        eta[mask], phi[mask],
        s=pt[mask] * 5, c=np.log1p(pt[mask]),
        cmap='viridis', alpha=0.7, edgecolors='black', linewidths=0.3
    )
    axes[ax_idx].set_xlabel(r'$\eta$')
    axes[ax_idx].set_ylabel(r'$\phi$')
    axes[ax_idx].set_title(f'{class_names[cls]} ({mask.sum()} particles)')
    plt.colorbar(sc, ax=axes[ax_idx], label=r'$\log(1 + p_T)$')

plt.suptitle('Jets in the $\\eta$-$\\phi$ plane (dot size $\\propto p_T$)', fontsize=14)
plt.tight_layout()
plt.show()

print("In these examples, the QCD jet has one loose core plus soft wide-angle particles,")
print("while Hbb, Tbqq, and H4q show harder multi-prong structure. That is the pattern")
print("the classifier will try to learn from particle coordinates and momenta alone.")

### Preparing the Data

The dataloader returns particles in shape `(jets, 4, 128)`. We transpose to `(jets, 128, 4)` so each particle is a 4-feature vector, and create a **mask** to distinguish real particles from zero-padding.

For a clear demo we keep just **two** well-separated classes -- QCD vs top->bqq -- and relabel them 0/1. With only two classes this tiny model actually learns, so the results are easy to read. Training still finishes in seconds.

This exact small binary task is reused in Notebook 2, so the Linformer comparison is matched to the baseline you build here.

In [ ]:
from tutorial_data import prepare_small_binary_task

task = prepare_small_binary_task(x_particles, y, n_use=6000, seed=42)
binary_classes = task['binary_classes']
binary_names = task['binary_names']

x_train = task['x_train']
mask_train = task['mask_train']
y_train = task['y_train']

x_test = task['x_test']
mask_test = task['mask_test']
y_test = task['y_test']

print(f"Binary task: {binary_names[0]} (0) vs {binary_names[1]} (1)")
print(f"Train: {len(x_train)} jets   Test: {len(x_test)} jets")
print(f"Train class balance: {int((y_train==0).sum())} {binary_names[0]}, {int((y_train==1).sum())} {binary_names[1]}")
print("Features standardized (zero mean, unit variance) using training statistics.")

---
## Section 3: The Building Blocks

We build the transformer piece by piece. Each component follows the same structure:
1. The idea in plain language
2. The formula
3. A short exercise (fill in 1-3 lines)
4. Solution + test cell

### 3a. Input Projection

Each particle starts as four raw numbers: $p_T$, $\eta$, $\phi$, and energy. Before attention can compare particles, we first map each 4-feature vector into a learned 32-dimensional representation.

A linear layer turns each 4-feature particle vector into a 32-dimensional embedding:

$$z = xW^T + b$$

where $x \in \mathbb{R}^4$, $W \in \mathbb{R}^{32 \times 4}$, $b \in \mathbb{R}^{32}$, and $z \in \mathbb{R}^{32}$. PyTorch's `nn.Linear` does this internally, but here we will write the matrix multiplication ourselves once so the shapes are explicit.

**Exercise 1:** Implement the linear projection by hand.

The learnable weight matrix `self.W` and bias `self.b` are already created for you. Complete the forward pass using $xW^T + b$. The test below checks whether a batch shaped `(2, 128, 4)` comes out as `(2, 128, 32)`.

In [ ]:
class ParticleEmbedding(nn.Module):
    """Projects particle features into a higher-dimensional space."""
    def __init__(self, input_dim, embed_dim):
        super().__init__()
        self.W = nn.Parameter(torch.randn(embed_dim, input_dim) * 0.01)
        self.b = nn.Parameter(torch.zeros(embed_dim))
    
    def forward(self, x):
        # TODO: apply z = x W^T + b
        return ...

<details>
<summary><b>Click for solution</b></summary>

The completed solution is in the minimized code cell below.

</details>

In [ ]:
class ParticleEmbedding(nn.Module):
    """Projects particle features into a higher-dimensional space."""
    def __init__(self, input_dim, embed_dim):
        super().__init__()
        self.W = nn.Parameter(torch.randn(embed_dim, input_dim) * 0.01)
        self.b = nn.Parameter(torch.zeros(embed_dim))
    
    def forward(self, x):
        return x @ self.W.T + self.b

In [ ]:
# Test: does the shape come out right?
embed = ParticleEmbedding(input_dim=4, embed_dim=32)
test_in = torch.randn(2, 128, 4)   # 2 jets, 128 particles, 4 features
test_out = embed(test_in)
print(f"Input:  {test_in.shape}")
print(f"Output: {test_out.shape}")
assert test_out.shape == (2, 128, 32), "Wrong shape!"
print("Passed.")

### 3b. Attention -- The Core Idea

Self-attention is the mechanism that distinguishes transformers from other architectures. Each particle computes a weighted sum over all other particles, where the weights are determined by learned compatibility functions.

#### Query, Key, Value

Each particle's embedding is projected into three separate vectors:
- **Query (Q):** what this particle is looking for
- **Key (K):** what this particle advertises about itself
- **Value (V):** the information this particle contributes

The attention score between particles $i$ and $j$ is the dot product of $i$'s query with $j$'s key -- a measure of how relevant $j$ is to $i$.

#### The Formula

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

Step by step:
1. **Scores** = $QK^T$ -- dot product between all query-key pairs
2. **Scale** by $\frac{1}{\sqrt{d_k}}$ -- prevents the dot products from growing too large in magnitude
3. **Softmax** -- normalizes each row into a probability distribution (sums to 1)
4. **Weighted sum** -- multiply by $V$ to produce the output

#### Masking

Jets have variable numbers of particles, padded to 128. To prevent real particles from attending to padding, we set the scores at padded positions to $-\infty$ before the softmax. Since $e^{-\infty} = 0$, padding receives zero attention weight.

**Exercise 2:** Fill in the three key lines of the attention computation.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Compute scaled dot-product attention.
    
    Args:
        Q: (batch, n_heads, seq_len, d_k)
        K: (batch, n_heads, seq_len, d_k)
        V: (batch, n_heads, seq_len, d_k)
        mask: (batch, 1, 1, seq_len) -- True for real, False for padding
    Returns:
        output, attention_weights
    """
    d_k = Q.size(-1)
    
    # TODO Step 1: Compute scores = Q @ K^T, then divide by sqrt(d_k)
    scores = ...
    
    # Apply mask (set padded positions to -inf before softmax)
    if mask is not None:
        scores = scores.masked_fill(~mask, float('-inf'))
    
    # TODO Step 2: Apply softmax over the last dimension
    attn_weights = ...
    attn_weights = torch.nan_to_num(attn_weights, nan=0.0)  # handle all-masked rows
    
    # TODO Step 3: Multiply attention weights by V
    output = ...
    
    return output, attn_weights

<details>
<summary><b>Click for solution</b></summary>

The completed solution is in the minimized code cell below.

</details>

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Compute scaled dot-product attention."""
    d_k = Q.size(-1)
    
    # Step 1: Compute scaled scores
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    # Apply mask
    if mask is not None:
        scores = scores.masked_fill(~mask, float('-inf'))
    
    # Step 2: Softmax -> attention weights (each row sums to 1)
    attn_weights = F.softmax(scores, dim=-1)
    attn_weights = torch.nan_to_num(attn_weights, nan=0.0)
    
    # Step 3: Weighted sum of values
    output = torch.matmul(attn_weights, V)
    
    return output, attn_weights

In [ ]:
# Test the attention function
Q = torch.randn(1, 1, 6, 8)   # 1 jet, 1 head, 6 particles, d_k=8
K = torch.randn(1, 1, 6, 8)
V = torch.randn(1, 1, 6, 8)

# Mask: first 4 are real, last 2 are padding
mask = torch.tensor([[[[True, True, True, True, False, False]]]])

out, weights = scaled_dot_product_attention(Q, K, V, mask)

print(f"Output shape: {out.shape}")
print(f"Weights shape: {weights.shape}")
print(f"\nAttention weights (row 0): {weights[0,0,0].tolist()}")
print(f"  -> Real positions sum to {weights[0,0,0,:4].sum():.4f} (should be 1.0)")
print(f"  -> Padded positions are {weights[0,0,0,4:].tolist()} (should be [0, 0])")
print("Passed.")

In [ ]:
# Visualize a small attention matrix
torch.manual_seed(0)
Q_v = torch.randn(1, 1, 5, 8)
K_v = torch.randn(1, 1, 5, 8)
V_v = torch.randn(1, 1, 5, 8)
_, w = scaled_dot_product_attention(Q_v, K_v, V_v)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(w[0, 0].detach().numpy(), cmap='Blues', vmin=0, vmax=0.5)
ax.set_xlabel('Key (which particle is being looked at)')
ax.set_ylabel('Query (which particle is looking)')
ax.set_title('Attention weights: who looks at whom?')
for i in range(5):
    for j in range(5):
        ax.text(j, i, f'{w[0,0,i,j]:.2f}', ha='center', va='center', fontsize=10)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print("Each row sums to 1.0 (probability distribution over keys).")

### 3c. Multi-Head Attention

A single attention head learns one set of Q, K, V projections and captures one type of particle relationship. Multi-head attention runs $h$ heads in parallel, each with its own projections, so the model can attend to different types of relationships simultaneously -- angular proximity in one head, momentum similarity in another.

In practice, we split $d_{\text{model}}$ evenly across heads. With $d_{\text{model}} = 32$ and 2 heads, each head operates on 16 dimensions.

**Exercise 3:** Fill in the reshape (splitting into heads) and the output projection.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
    
    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        
        # Project to Q, K, V
        Q = self.W_Q(x)  # (batch, seq_len, d_model)
        K = self.W_K(x)
        V = self.W_V(x)
        
        # TODO: Reshape Q from (batch, seq_len, d_model) to (batch, n_heads, seq_len, d_k)
        # Hint: use .view(...) then .transpose(1, 2)
        Q = ...
        K = ...
        V = ...
        
        # Reshape mask for broadcasting
        if mask is not None:
            mask = mask.unsqueeze(1).unsqueeze(2)  # (batch, 1, 1, seq_len)
        
        # Attention
        attn_out, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        # Reshape back: (batch, n_heads, seq_len, d_k) -> (batch, seq_len, d_model)
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        
        # TODO: Apply the output projection W_O
        output = ...
        
        return output, attn_weights

<details>
<summary><b>Click for solution</b></summary>

The completed solution is in the minimized code cell below.

</details>

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
    
    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        
        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)
        
        # Split into heads: (batch, seq_len, d_model) -> (batch, n_heads, seq_len, d_k)
        Q = Q.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        
        if mask is not None:
            mask = mask.unsqueeze(1).unsqueeze(2)
        
        attn_out, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        # Merge heads back
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        
        # Output projection
        output = self.W_O(attn_out)
        
        return output, attn_weights

In [ ]:
# Test multi-head attention
mha = MultiHeadAttention(d_model=32, n_heads=2)
test_x = torch.randn(2, 10, 32)
test_mask = torch.ones(2, 10, dtype=torch.bool)
test_mask[:, 7:] = False

out, w = mha(test_x, test_mask)
print(f"Input:  {test_x.shape}")
print(f"Output: {out.shape}")
print(f"Attention: {w.shape}  (batch, {w.shape[1]} heads, seq, seq)")
print(f"Attention to padded (should be 0): {w[0, 0, 0, 7:].tolist()}")
print("Passed.")

### 3d. Feed-Forward Network, LayerNorm, and Residual Connections

After attention mixes information between particles, a two-layer feed-forward network (with GELU activation) processes each particle independently.

Two standard components keep training stable:

- **LayerNorm:** normalizes features per particle to prevent values from blowing up or vanishing.
- **Residual connections:** `output = input + sublayer(input)`. The skip connection gives gradients a direct path through deep networks.

No exercise here -- read the code below.

In [ ]:
class FeedForward(nn.Module):
    """A small neural network applied independently to each particle."""
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)    # expand
        self.linear2 = nn.Linear(d_ff, d_model)    # compress back
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        x = self.linear1(x)      # (batch, seq, d_model) -> (batch, seq, d_ff)
        x = F.gelu(x)            # smooth activation function
        x = self.dropout(x)
        x = self.linear2(x)      # (batch, seq, d_ff) -> (batch, seq, d_model)
        return x

# Quick test
ffn = FeedForward(d_model=32, d_ff=64)
test_ffn = torch.randn(2, 10, 32)
print(f"FFN: {test_ffn.shape} -> {ffn(test_ffn).shape}")
print(f"Parameters: {sum(p.numel() for p in ffn.parameters()):,}")

### 3e. Putting It All Together: The Transformer Block

A transformer block has two sub-steps.

First, layer normalization prepares the particle embeddings for multi-head attention. The attention output is then added back to the original input through a residual connection.

Second, another layer normalization prepares those updated embeddings for the feed-forward network. The feed-forward output is again added back through a residual connection.

So the block makes two controlled updates: one from particle-to-particle communication, and one from a per-particle nonlinear transformation. Stacking multiple blocks lets the model refine the jet representation step by step.

In [ ]:
class TransformerBlock(nn.Module):
    """One transformer encoder block (pre-norm variant)."""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.dropout1 = nn.Dropout(dropout)
        
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.dropout2 = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Attention block with residual
        x_norm = self.norm1(x)
        attn_out, attn_weights = self.attn(x_norm, mask)
        x = x + self.dropout1(attn_out)       # residual connection
        
        # FFN block with residual
        x_norm = self.norm2(x)
        ffn_out = self.ffn(x_norm)
        x = x + self.dropout2(ffn_out)         # residual connection
        
        return x, attn_weights

# Test
block = TransformerBlock(d_model=32, n_heads=2, d_ff=64)
test_in = torch.randn(2, 10, 32)
test_mask = torch.ones(2, 10, dtype=torch.bool)
out, w = block(test_in, test_mask)
print(f"TransformerBlock: {test_in.shape} -> {out.shape}")
print(f"Parameters: {sum(p.numel() for p in block.parameters()):,}")
print("Passed.")

---
## Section 4: Building and Training Our Jet Classifier

The full model:

1. **Embed** each particle: 4 features $\to$ 32 dimensions
2. **Transform** with 2 stacked transformer blocks
3. **Pool** over all particles (mean pooling) $\to$ one vector per jet
4. **Classify** with a linear layer $\to$ 2 class scores (QCD vs top)

Mean pooling is permutation invariant (the average does not depend on ordering), so combined with our permutation-equivariant transformer, the entire model is permutation invariant.

The model is deliberately tiny: $d_{\text{model}}=32$, 2 heads, 2 layers, $d_{ff}=64$. It trains in seconds.

### Exercise 4: Masked mean pooling

The transformer blocks output one 32-dim vector per particle, so a jet is still a *set* of up to 128 vectors. The classifier needs a **single** vector per jet. The standard way to collapse a set into one vector is to **average** its elements -- this is *pooling*. Averaging is a natural choice here because it is permutation-invariant (reordering the particles does not change the mean), which matches the fact that a jet is an unordered set.

The catch is padding. Every jet is zero-padded out to 128 slots, but most jets have far fewer real particles. A plain `x.mean(dim=1)` divides by 128 every time -- so for a jet with, say, 10 real particles it adds up the 10 real vectors plus 118 zeros and divides by 128. That shrinks the real signal by more than 10x, and how much it shrinks depends on how much padding the jet happens to have. That is noise we put in ourselves.

A **masked mean** fixes this: sum only the real particle vectors, and divide by the number of real particles.

$$h_{\text{jet}} = \frac{\sum_i m_i\, x_i}{\sum_i m_i}, \qquad m_i = \begin{cases}1 & \text{real particle}\\ 0 & \text{padding}\end{cases}$$

In the code, `m = mask.unsqueeze(-1)` is that per-particle 0/1 indicator (shape `(batch, 128, 1)` so it broadcasts over the 32 feature dimensions). Multiplying `x * m` zeroes out the padded rows; summing the numerator and dividing by `m.sum()` -- the real-particle count -- gives a true average over real particles only.

**Your task:** fill in the two lines.
- `summed`: the sum of the real particle vectors. Zero out padding with `m`, then sum over the particle axis (`dim=1`).
- `count`: the number of real particles per jet. Sum `m` over `dim=1`; wrap it in `.clamp(min=1)` so a jet that is somehow all padding never divides by zero.

In [ ]:
def masked_mean(x, mask):
    """Average particle vectors over the REAL particles only.

    x:    (batch, n_particles, d_model)
    mask: (batch, n_particles)   True = real particle, False = padding
    returns: (batch, d_model)    one pooled vector per jet
    """
    m = mask.unsqueeze(-1).float()        # (batch, n_particles, 1): 1 for real, 0 for padding

    # TODO (2 lines):
    #   summed = sum of the REAL particle vectors  (use m to zero out padding, then sum over dim=1)
    #   count  = number of real particles per jet  (sum m over dim=1; .clamp(min=1) avoids divide-by-zero)
    summed = ...
    count = ...

    return summed / count

<details>
<summary><b>Click for solution</b></summary>

The completed solution is in the minimized code cell below.

</details>

In [ ]:
def masked_mean(x, mask):
    """Average particle vectors over the real particles only."""
    m = mask.unsqueeze(-1).float()
    summed = (x * m).sum(dim=1)            # padded slots contribute 0
    count = m.sum(dim=1).clamp(min=1)      # how many real particles (never divide by 0)
    return summed / count

# Test on a toy jet: 2 real particles (values 1 and 3) and 1 padded slot.
xt = torch.tensor([[[1., 1.], [3., 3.], [0., 0.]]])
mt = torch.tensor([[True, True, False]])
print("masked_mean:", masked_mean(xt, mt).tolist(), " (correct: mean of 1 and 3 = 2.0)")
print("naive mean :", xt.mean(dim=1).tolist(), " (wrong: divides by 3, counts the padding)")
assert torch.allclose(masked_mean(xt, mt), torch.tensor([[2., 2.]]))
print("Passed -- padding is correctly ignored.")

In [ ]:
class JetClassifier(nn.Module):
    """A tiny transformer for classifying particle jets."""
    def __init__(self, input_dim=4, d_model=32, n_heads=2, d_ff=64,
                 n_layers=2, n_classes=2, dropout=0.1):
        super().__init__()
        
        # 1. Input embedding
        self.embedding = ParticleEmbedding(input_dim, d_model)
        
        # 2. Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        
        # 3. Final norm + classifier
        self.final_norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, n_classes)
        
        # For storing attention weights (used later for visualization)
        self.attention_maps = []
    
    def forward(self, x, mask=None):
        # Embed particles
        x = self.embedding(x)          # (batch, 128, 4) -> (batch, 128, 32)
        
        # Pass through transformer blocks
        self.attention_maps = []
        for block in self.blocks:
            x, attn_w = block(x, mask)
            self.attention_maps.append(attn_w)
        
        # Final norm
        x = self.final_norm(x)
        
        # Mean pooling over particles, ignoring padding -- uses masked_mean from Exercise 4
        x = masked_mean(x, mask) if mask is not None else x.mean(dim=1)
        # x is now (batch, 32) -- one vector per jet
        
        # Classify
        logits = self.classifier(x)    # (batch, 2)
        return logits


model = JetClassifier(
    input_dim=4, d_model=32, n_heads=2, d_ff=64, n_layers=2, n_classes=2
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model has {n_params:,} parameters")
print(f"\nArchitecture:")
print(f"  Input: 4 -> d_model={32}")
print(f"  Transformer: {2} layers, {2} heads, d_ff={64}")
print(f"  Output: {32} -> 2 classes (QCD vs top)")

In [ ]:
# Create data loaders
batch_size = 256

train_loader = DataLoader(
    TensorDataset(x_train, mask_train, y_train),
    batch_size=batch_size, shuffle=True
)
test_loader = DataLoader(
    TensorDataset(x_test, mask_test, y_test),
    batch_size=batch_size, shuffle=False
)

print(f"Train: {len(train_loader)} batches of {batch_size}")
print(f"Test:  {len(test_loader)} batches")

In [ ]:
# Train for 10 epochs
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

n_epochs = 10
train_losses = []
test_losses = []
test_accs = []

for epoch in range(n_epochs):
    # Train
    model.train()
    epoch_loss = 0
    n_batches = 0
    for bx, bm, by in train_loader:
        bx, bm, by = bx.to(device), bm.to(device), by.to(device)
        optimizer.zero_grad()
        logits = model(bx, bm)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    train_losses.append(epoch_loss / n_batches)
    
    # Evaluate
    model.eval()
    test_loss = 0
    correct = 0
    total = 0
    n_test = 0
    with torch.no_grad():
        for bx, bm, by in test_loader:
            bx, bm, by = bx.to(device), bm.to(device), by.to(device)
            logits = model(bx, bm)
            test_loss += criterion(logits, by).item()
            n_test += 1
            correct += (logits.argmax(1) == by).sum().item()
            total += by.size(0)
    test_losses.append(test_loss / n_test)
    test_accs.append(correct / total)
    
    print(f"Epoch {epoch+1}/{n_epochs}  "
          f"Train loss: {train_losses[-1]:.4f}  "
          f"Test loss: {test_losses[-1]:.4f}  "
          f"Test acc: {test_accs[-1]:.3f}")

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

epochs = range(1, n_epochs + 1)
ax1.plot(epochs, train_losses, 'b-o', label='Train')
ax1.plot(epochs, test_losses, 'r-o', label='Test')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.axhline(np.log(2), color='gray', ls='--', alpha=0.5, label='Random guess (ln 2)')
ax1.set_title('Training & Test Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, test_accs, 'g-o')
ax2.axhline(0.5, color='gray', ls='--', alpha=0.5, label='Random (50%)')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Test Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal test accuracy: {test_accs[-1]:.1%}")
print(f"Random baseline: 50.0% (2 classes)")
print(f"\nTwo well-separated classes, so even this tiny model learns a useful decision boundary.")

### How to read these curves

The dashed lines are the two baselines for a balanced binary task: loss near ln 2 means the model is guessing, and accuracy near 0.5 means it is flipping a coin.

From epochs 1 to 5, both held-out metrics move in the right direction. The model is learning features that separate QCD from top jets, not just memorizing the training sample.

After that, the story gets more realistic. The training loss keeps sliding downward, but the held-out loss stops improving cleanly and bumps upward around epochs 6 and 8 to 10. The held-out accuracy peaks around 0.74 and then settles a bit lower. That is a mild overfitting signal, which is exactly the kind of behavior you want to learn to recognize early.

For this tutorial, that is a useful regime: the model is simple enough that the curves are easy to interpret, but strong enough to learn a real particle-physics classification task.

### Why this small task is useful

We use **two well-separated classes** here so the notebook teaches the mechanics of self-attention on jets, not the frustration of a benchmark that is too hard for a toy model. On the full 10-class JetClass problem, this network is much too small to be competitive. On QCD vs top, it is just large enough to learn something visible and interpretable.

That is the right scale for a tutorial in particle physics. You can watch the optimizer improve the classifier, inspect the attention maps, and keep the whole architecture in your head. Notebook 3 then shows what changes when the same basic ingredients are given a stronger physics prior.

---
## Section 5: A Look at the Attention Weights

Since the attention matrix $A_{ij}$ is computed explicitly, we can inspect which particles the model relates. This is one of the practical advantages of transformer architectures -- the learned relationships are directly visible.

We explore this more thoroughly in Notebook 3, where recent interpretability work (Wang et al., [arXiv:2412.03673](https://arxiv.org/abs/2412.03673)) shows that trained Particle Transformers learn sparse, physically meaningful attention patterns.

In [ ]:
# Get predictions on the test set
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for bx, bm, by in test_loader:
        bx, bm = bx.to(device), bm.to(device)
        logits = model(bx, bm)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(by.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

print(f"Overall accuracy: {(all_preds == all_labels).mean():.1%}")

In [ ]:
# Visualize attention matrices for two different jet types
model.eval()

fig, axes = plt.subplots(2, 2, figsize=(13, 11))

viz_classes = [0, 1]  # QCD, top (binary labels)
for row, cls in enumerate(viz_classes):
    # Find a correctly classified jet of this type
    matches = np.where((all_labels == cls) & (all_preds == cls))[0]
    if len(matches) == 0:
        matches = np.where(all_labels == cls)[0]
    idx = matches[0]
    
    jet_x = x_test[idx:idx+1].to(device)
    jet_mask = mask_test[idx:idx+1].to(device)
    with torch.no_grad():
        _ = model(jet_x, jet_mask)
    
    n_real = int(mask_test[idx].sum())
    n_show = min(n_real, 30)  # show at most 30 for readability
    
    for col, layer_idx in enumerate([0, 1]):
        # Average over heads
        attn = model.attention_maps[layer_idx][0].mean(dim=0).cpu().numpy()
        attn_sub = attn[:n_show, :n_show]
        
        im = axes[row, col].imshow(attn_sub, cmap='viridis', aspect='auto')
        axes[row, col].set_title(f'{binary_names[cls]} -- Layer {layer_idx+1}')
        axes[row, col].set_xlabel('Key particle')
        axes[row, col].set_ylabel('Query particle')
        plt.colorbar(im, ax=axes[row, col])

plt.suptitle('Attention patterns for different jet types\n'
             '(showing first 30 particles, averaged over heads)', fontsize=13)
plt.tight_layout()
plt.show()

print("Read these heatmaps column-wise: a bright vertical band means many query particles")
print("look to the same key particle. There is no positional encoding here, so the exact")
print("column number is not special by itself. The useful signal is that attention becomes")
print("concentrated on a few particles, and that the concentration pattern differs by jet type.")

---
## Section 6: Summary and What's Next

### What we built

A transformer jet classifier from scratch:

| Component | Role |
|:---|:---|
| Input projection | 4 features per particle $\to$ 32-dimensional embedding |
| Self-attention | Each particle computes a weighted sum over all other particles |
| Multi-head attention | Parallel attention heads for different relationship types |
| Feed-forward network | Per-particle nonlinear transformation after attention |
| LayerNorm + Residuals | Training stability for deep networks |
| Mean pooling | Permutation-invariant aggregation to a single jet vector |

### Takeaways

1. Transformers (without positional encoding) are permutation equivariant, which makes them a natural fit for unordered jet data.
2. Attention weights are directly inspectable -- we can see which particles the model relates.

### Next

- **Notebook 2:** The $O(n^2)$ cost of attention and how to reduce it (Linformer)
- **Notebook 3:** The Particle Transformer -- encoding physics into attention, plus interpretability

### References

1. Vaswani et al., "Attention Is All You Need", NeurIPS 2017. [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
2. Qu et al., "Particle Transformer for Jet Tagging", ICML 2022. [arXiv:2202.03772](https://arxiv.org/abs/2202.03772)
3. Wang et al., "Interpreting Transformers for Jet Tagging", NeurIPS ML4PS 2024. [arXiv:2412.03673](https://arxiv.org/abs/2412.03673)